---
short_title: Fehlerbehandlung & Performance
numbering:
    heading_1: true
    heading_2: true
    title: true
---

# Fehlerbehandlung und Performance

Echte Daten sind unvollständig und echte Dateien können groß sein. Die
Grundlagen der Fehlerbehandlung – `try`/`except`, `ValueError` und
`FileNotFoundError` – kennen Sie aus
[Projekt CSV I – Datenvalidierung](../../040-Projekt_CSV_I/045-Datenvalidierung.ipynb)
und [Dateien öffnen](../../040-Projekt_CSV_I/020-Dateien_öffnen.ipynb). Hier
bauen wir darauf auf und kümmern uns um

- die Ergänzungen `else`, `finally` und `raise`,
- das **speicherschonende Verarbeiten** großer Dateien (Streaming),
- das **Messen und Verbessern** der Laufzeit sowie
- die **Migration** von Rohdaten in eine Zielstruktur.

:::{seealso} Vorwissen
:icon: false

Die Kapitel
[Erweiterte CSV-Verarbeitung](010-Erweiterte_CSV_Verarbeitung.ipynb) und
[Manuelle Analyse](020-Manuelle_Analyse.ipynb) werden vorausgesetzt.
:::


## `else`, `finally` und `raise`

`try`/`except` haben Sie bereits kennengelernt. Drei Ergänzungen machen die
Fehlerbehandlung vollständig:

- `else` läuft nur, wenn **kein** Fehler auftrat.
- `finally` läuft **immer** – gut zum Aufräumen.
- `raise` wirft selbst einen Fehler, z. B. nach einer Plausibilitätsprüfung.


In [1]:
def quotient(a, b):
    try:
        ergebnis = a / b
    except ZeroDivisionError:
        print("Division durch Null ist nicht erlaubt.")
        return None
    else:
        return ergebnis
    finally:
        print(f"Versuch: {a} / {b}")

print(quotient(10, 2))
print(quotient(10, 0))

Versuch: 10 / 2
5.0
Division durch Null ist nicht erlaubt.
Versuch: 10 / 0
None


In [2]:
def parse_jahr(text):
    """Wandelt Text in ein Jahr um und prüft den plausible Bereich."""
    jahr = int(text)
    if not 1000 <= jahr <= 2100:
        raise ValueError(f"Ungültiges Jahr: {jahr}")
    return jahr

try:
    parse_jahr("0")
except ValueError as fehler:
    print("Fehler:", fehler)

Fehler: Ungültiges Jahr: 0


## Fehlerhafte Zeilen protokollieren

Wie Sie fehlerhafte Zeilen sammeln, statt sie stillschweigend zu überspringen,
haben Sie in
[Projekt CSV I – Datenvalidierung](../../040-Projekt_CSV_I/045-Datenvalidierung.ipynb)
gesehen. Hier wenden wir dasselbe Muster auf den `csv.DictReader` an, der die
Spalten bereits über ihre Namen anspricht.


In [3]:
import csv
import io

rohdaten = """isbn,year,sales
9780241951446,1984,61
9783518587249,1999,kaputt
9780141185422,1922,15
9783518587157,,15
"""

gute_zeilen = []
fehlerhafte_zeilen = []

for nummer, zeile in enumerate(csv.DictReader(io.StringIO(rohdaten)), start=2):
    try:
        zeile["year"] = int(zeile["year"])
        zeile["sales"] = int(zeile["sales"])
        gute_zeilen.append(zeile)
    except ValueError as fehler:
        fehlerhafte_zeilen.append((nummer, zeile, str(fehler)))

print("Gültige Zeilen:", len(gute_zeilen))
print("Fehlerhafte Zeilen:")
for nummer, zeile, fehler in fehlerhafte_zeilen:
    print(f"  Zeile {nummer}: {zeile} -> {fehler}")

Gültige Zeilen: 2
Fehlerhafte Zeilen:
  Zeile 3: {'isbn': '9783518587249', 'year': 1999, 'sales': 'kaputt'} -> invalid literal for int() with base 10: 'kaputt'
  Zeile 5: {'isbn': '9783518587157', 'year': '', 'sales': '15'} -> invalid literal for int() with base 10: ''


## Große Dateien zeilenweise verarbeiten (Streaming)

`f.readlines()` liest **alle** Zeilen auf einmal in den Speicher. Bei großen
Dateien kann das viel Arbeitsspeicher kosten. Eine Datei ist aber auch
**iterierbar**: Eine `for`-Schleife holt sich jeweils nur die nächste Zeile.
Man nennt das **Streaming**.

Wir erzeugen zuerst eine künstlich große Datei in einem temporären Verzeichnis
– so bleibt das Projekt sauber.

In [4]:
import tempfile
import time
from pathlib import Path

# Große Testdatei in einem temporären Verzeichnis erzeugen
tmp_dir = tempfile.mkdtemp()
grosse_datei = Path(tmp_dir) / "gross.csv"
with open(grosse_datei, "w", encoding="utf-8", newline="") as f:
    f.write("id,wert\n")
    for i in range(200_000):
        f.write(f"{i},{i % 97}\n")

print("Dateigröße:", round(grosse_datei.stat().st_size / 1024 / 1024, 2), "MiB")

Dateigröße: 1.78 MiB


In [5]:
def zaehle_mit_liste(pfad):
    with open(pfad, encoding="utf-8") as f:
        zeilen = f.readlines()      # alle Zeilen gleichzeitig im Speicher
    return len(zeilen)

def zaehle_mit_stream(pfad):
    anzahl = 0
    with open(pfad, encoding="utf-8") as f:
        for _ in f:                 # eine Zeile nach der anderen
            anzahl += 1
    return anzahl

print("Gezaehlt (Streaming):", zaehle_mit_stream(grosse_datei))

Gezaehlt (Streaming): 200001


## Laufzeit messen

„Schneller“ ist kein Gefühl, sondern eine Messung. Zwei Werkzeuge sind
praktisch:

- `time.perf_counter()` – eine sehr genaue Uhr für einzelne Messungen.
- `timeit` – wiederholt eine kleine Anweisung und mittelt die Laufzeit.

In [6]:
def messe(funktion, *argumente):
    """Misst die Laufzeit einer Funktion in Sekunden."""
    start = time.perf_counter()
    ergebnis = funktion(*argumente)
    dauer = time.perf_counter() - start
    return ergebnis, dauer

erg1, dauer_liste = messe(zaehle_mit_liste, grosse_datei)
erg2, dauer_stream = messe(zaehle_mit_stream, grosse_datei)
print(f"readlines: {dauer_liste:.4f} s ({erg1} Zeilen)")
print(f"Streaming: {dauer_stream:.4f} s ({erg2} Zeilen)")

readlines: 0.0061 s (200001 Zeilen)
Streaming: 0.0067 s (200001 Zeilen)


In [7]:
import timeit

# Mittelwert: selbst geschrieben vs. statistics.mean
setup = "from statistics import mean; daten = list(range(180))"
dauer_selbst = timeit.timeit("sum(daten) / len(daten)", setup=setup, number=100_000)
dauer_mean = timeit.timeit("mean(daten)", setup=setup, number=100_000)
print(f"sum()/len():      {dauer_selbst:.4f} s")
print(f"statistics.mean:  {dauer_mean:.4f} s")

sum()/len():      0.0357 s
statistics.mean:  1.4423 s


## Einfache Optimierungen

Kleine Änderungen an der Datenstruktur können viel bewirken. Ein Beispiel:
Eine **Mitgliedschaftsprüfung** (`in`) durchsucht eine Liste von vorne nach
hinten – bei einer Menge (`set`) geht das fast in konstanter Zeit.

In [8]:
grosse_liste = list(range(100_000))
grosse_menge = set(grosse_liste)

dauer_liste = timeit.timeit("99_999 in grosse_liste", globals=globals(), number=200)
dauer_menge = timeit.timeit("99_999 in grosse_menge", globals=globals(), number=200)
print(f"Liste: {dauer_liste:.4f} s")
print(f"Menge: {dauer_menge:.4f} s")

Liste: 0.1070 s
Menge: 0.0000 s


Weitere Faustregeln:

- Nur das Nötige in den `try`-Block schreiben.
- Ergebnisse, die sich nicht ändern, nur einmal berechnen (z. B. `len()`).
- Für häufige `in`-Prüfungen ein `set` statt einer `list` verwenden.
- Erst messen, dann optimieren – nicht umgekehrt.

Zum Aufräumen löschen wir das temporäre Verzeichnis.

In [9]:
import shutil
shutil.rmtree(tmp_dir)
print("Temporäres Verzeichnis entfernt.")

Temporäres Verzeichnis entfernt.


## Datenmigration und -transformation

Am Ende eines Projekts steht oft die Überführung von Rohdaten in eine neue,
besser nutzbare Struktur – eine **Migration**. Das Muster kennen Sie schon:
Alte Schlüssel werden auf neue abgebildet, Text wird in Zahlen umgewandelt.

In [10]:
alte_saetze = [
    {"name": "Murakami, Haruki", "yr": "1984", "n": "61"},
    {"name": "Boltanski, Luc", "yr": "1999", "n": "40"},
]

def migriere(alt):
    """Überführt einen alten Datensatz in die neue Zielstruktur."""
    return {
        "author": alt["name"],
        "year": int(alt["yr"]),
        "sales": int(alt["n"]),
    }

neue_saetze = [migriere(alt) for alt in alte_saetze]
for satz in neue_saetze:
    print(satz)

{'author': 'Murakami, Haruki', 'year': 1984, 'sales': 61}
{'author': 'Boltanski, Luc', 'year': 1999, 'sales': 40}


## Übung: Fehlertoleranter Import 💪

:::{exercise} Import mit Fehlerprotokoll
:label: import-fehlerprotokoll

Schreiben Sie eine Funktion `importiere(rohtext)`, die mit `csv.DictReader`
über die folgenden Daten läuft und eine Liste typisierter Dictionaries
zurückgibt. Fehlerhafte Zeilen (nicht konvertierbare `sales`-Werte) sollen
gesammelt und am Ende als Liste zurückgegeben werden. Die Funktion soll also
`(gute, fehlerhafte)` zurückgeben.

```python
rohtext = (
    "author,sales\n"
    "Murakami, Haruki,61\n"   # Achtung: zusätzliches Feld/Leerzeichen
    "Kafka, Franz,19\n"
    "Boltanski, Luc,oops\n"
)
```

Hinweis: Nutzen Sie `try`/`except ValueError` und prüfen Sie ggf. auch die
Feldanzahl.
:::

:::{solution} import-fehlerprotokoll
:class: dropdown

```python
import csv
import io

def importiere(rohtext):
    gute, fehlerhafte = [], []
    for nummer, zeile in enumerate(csv.DictReader(io.StringIO(rohtext)), start=2):
        try:
            satz = {
                "author": zeile["author"].strip(),
                "sales": int(zeile["sales"].strip()),
            }
        except (ValueError, AttributeError) as fehler:
            fehlerhafte.append((nummer, zeile, str(fehler)))
        else:
            gute.append(satz)
    return gute, fehlerhafte

rohtext = (
    "author,sales\n"
    "Murakami, Haruki,61\n"
    "Kafka, Franz,19\n"
    "Boltanski, Luc,oops\n"
)
gute, fehlerhafte = importiere(rohtext)
print("Gut:", gute)
print("Fehlerhaft:", fehlerhafte)
```

Hinweis: Die Zeile `Murakami, Haruki,61` enthält ein Komma im Autor\*innen-Feld
und wird deshalb nicht korrekt getrennt – ein schönes Beispiel dafür, dass
Anführungszeichen in der Quelle nötig gewesen wären.
:::

## Zusammenfassung

- `try`/`except` fängt vorhersehbare Fehler ab; `else`/`finally`/`raise`
  ergänzen das Verhalten.
- Häufige Fehler: `ValueError` (Umwandlung) und `FileNotFoundError` (Datei).
- Fehlerhafte Zeilen können **protokolliert** statt still übersprungen werden.
- Dateien werden **zeilenweise** verarbeitet; das spart Speicher bei großen
  Dateien.
- `time.perf_counter()` und `timeit` machen Laufzeiten messbar; ein `set`
  beschleunigt Mitgliedschaftsprüfungen.
- **Datenmigration** überführt Rohdaten in eine Zielstruktur.